In [1]:
# ==============================================================================
# STAGE 3: DIRECT PREFERENCE OPTIMIZATION (DPO ALIGNMENT WORKFLOW)
# ==============================================================================

# ------------------------------------------------------------------------------
# PRE-REQUISITE DEPENDENCY INSTALLATION (One-by-One Approach)
# ------------------------------------------------------------------------------
!pip -q install unsloth
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets
!pip install peft --no-deps
!pip install trl --no-deps
!pip install accelerator --no-deps
!pip install bitsandbytes --no-deps
!pip install xformers --no-deps

import os
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import DPOTrainer

# CRITICAL: Unsloth requires patching the DPOTrainer before importing or running it
PatchDPOTrainer()

# ------------------------------------------------------------------------------
# 1. LOADING THE SFT MODEL
# ------------------------------------------------------------------------------
print("Step 1: Loading the instruction-tuned SFT model and matching adapter layers...")
max_seq_length = 2048

# Load the adapter weights created during Stage 2
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/domain-ai-assistant-finetuning/models/stage2_sft_adapter",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Safely reuse or wrap the current active PEFT matrix configurations
if hasattr(model, "peft_config") or hasattr(model, "active_adapters"):
    print("✨ Detected active Stage 2 SFT adapters attached to the model layout.")
    try:
        # Scale parameters inside the configuration dictionary natively for contrastive tuning
        for adapter_name, config in model.peft_config.items():
            config.lora_alpha = 32  # Keep aligned with SFT configuration for stability
        print("✅ Active adapter configuration balanced and locked for preference optimization.")
    except Exception:
        print("ℹ️ Reusing current base active adapter weights directly.")
else:
    print("🔄 Initializing fresh PEFT adapters container for preference tuning...")
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 32,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

# ------------------------------------------------------------------------------
# 2 & 3. LOADING AND FORMATTING THE PREFERENCE DATASET
# ------------------------------------------------------------------------------
print("\nStep 2 & 3: Loading and mapping the preference dataset...")

# Define the exact instruction prompt template matching Stage 2 for formatting symmetry
def format_dpo_samples(example):
    return {
        "prompt": f"You are an expert customer support assistant. Provide clear, accurate, and structured answers.\n\n### Question:\n{example['prompt']}\n\n### Response:\n",
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

# Load the preference dataset JSONL
dataset = load_dataset("json", data_files={"train": "/content/drive/MyDrive/domain-ai-assistant-finetuning/data/preference_dataset.jsonl"})
dataset = dataset.map(format_dpo_samples)

# ------------------------------------------------------------------------------
# 4 & 5. CONFIGURING AND RUNNING DPO ALIGNMENT
# ------------------------------------------------------------------------------
print("\nStep 4 & 5: Initializing DPOTrainer and running alignment loop...")

# Import DPOConfig directly from trl to ensure all internal attributes exist
from trl import DPOConfig

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth optimizes memory by bypassing a separate explicit reference model
    args = DPOConfig( # Explicit configuration passed to 'args' parameter
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        max_steps = 50,
        learning_rate = 5e-6,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "/content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/stage3_dpo",
        report_to = "none"
    ), # <-- Correctly closing the DPOConfig object here
    beta = 0.1, # Implicit language model penalty constraint factor
    train_dataset = dataset["train"],
    tokenizer = tokenizer,
    max_length = max_seq_length,
    max_prompt_length = 512,
)

dpo_trainer.train()

# ------------------------------------------------------------------------------
# 6. SAVING THE DPO-ALIGNED MODEL
# ------------------------------------------------------------------------------
print("\nStep 6: Archiving final DPO-aligned model and tokenizer configurations...")
output_dpo_path = "/content/drive/MyDrive/domain-ai-assistant-finetuning/models/final_dpo_model"
model.save_pretrained(output_dpo_path)
tokenizer.save_pretrained(output_dpo_path)
print(f"💾 Production model successfully written to: '{output_dpo_path}'")

# ------------------------------------------------------------------------------
# 7. TESTING THE MODEL AFTER DPO
# ------------------------------------------------------------------------------
print("\n" + "="*60 + "\nStep 7: Executing Post-DPO Alignment Verification Inference...\n" + "="*60)

# Switch active parameters to fast decoding optimization kernels
FastLanguageModel.for_inference(model)

# Test query tracking a severe shipping delay scenario
test_query = "What is the policy for processing a refund if my package was delayed for weeks and missed my event?"

eval_prompt = f"""You are an expert customer support assistant. Provide clear, accurate, and structured answers.

### Question:
{test_query}

### Response:
"""

inputs = tokenizer([eval_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Isolate the newly aligned production output text
aligned_response = decoded_output.split("### Response:\n")[-1].strip()

print(f"Test Customer Prompt:\n-> {test_query}\n")
print(f"DPO Aligned Assistant Response:\n-> {aligned_response}\n")
print("="*60 + "\nStage 3 Preference Alignment Notebook Execution Complete!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth 2026.7.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Step 1: Loading the instruction-tuned SFT model and matching adapter layers...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: /content/drive/MyDrive/domain-ai-assistant-finetuning/models/stage2_sft_adapter had a bad pad_token (<|endoftext|>). Using pad_token = <|vision_pad|>.


Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✨ Detected active Stage 2 SFT adapters attached to the model layout.
✅ Active adapter configuration balanced and locked for preference optimization.

Step 2 & 3: Loading and mapping the preference dataset...

Step 4 & 5: Initializing DPOTrainer and running alignment loop...


...

Applying chat template to train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 8 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,17.121700,-28.701431,-11.579788,0.000000,-17.121643,-394.889221,-156.358566,-1.137899,-1.312438
2,16.202700,-28.370434,-12.167717,0.000000,-16.202715,-391.964233,-163.463699,-1.115984,-1.301236
3,19.061100,-28.251787,-9.190667,0.000000,-19.061123,-383.203003,-123.554245,-1.150166,-1.114413
4,19.217900,-29.522190,-10.304343,0.000000,-19.217848,-400.679321,-138.546875,-1.142481,-1.168185
5,18.076100,-28.873631,-10.797556,0.000000,-18.076073,-395.907166,-145.976074,-1.140000,-1.262314
6,17.351200,-27.058828,-9.707642,0.000000,-17.351189,-373.038330,-131.231262,-1.159225,-1.193603
7,17.276100,-25.235165,-7.959045,0.000000,-17.276119,-353.335693,-111.104813,-1.120561,-1.180478
8,18.545100,-28.088264,-9.543188,0.000000,-18.545078,-386.942169,-129.654648,-1.180421,-1.208152
9,15.395000,-25.651697,-10.256794,0.000000,-15.394904,-361.231476,-138.669586,-1.215084,-1.309184
10,9.956900,-22.104000,-12.147458,0.000000,-9.956541,-323.954834,-159.965973,-1.141227,-1.296550



Step 6: Archiving final DPO-aligned model and tokenizer configurations...
💾 Production model successfully written to: '/content/drive/MyDrive/domain-ai-assistant-finetuning/models/final_dpo_model'

Step 7: Executing Post-DPO Alignment Verification Inference...
Test Customer Prompt:
-> What is the policy for processing a refund if my package was delayed for weeks and missed my event?

DPO Aligned Assistant Response:
-> A delayed-transit gateway baseline was recorded, reversing the gateway line item weight. The duplicate duplicate payment barcode was replaced to an invalid barcode endpoint to replace() function. Your monthly charge card reconciliation balance is checked against the Balance Adjustments tab before being archived. A separate address barcodes are checked out to separate standard economy shipping weight folders to avoid duplicate regional delivery-transit-transit window headers. Separate standard economy weight profiles are checked to regenerate a unique secondary address ba